In [174]:
import numpy as np
def rmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def se(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt((y_true - y_pred) ** 2)

def mae(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs(y_true - y_pred))

def ae(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.abs(y_true - y_pred)

def r2_score(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    ss_res = np.sum((y_true - y_pred) ** 2)            # suma kwadratów reszt
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)   # całkowita wariancja

    return 1 - ss_res / ss_tot

def relative_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    eps = 1e-12
    return np.abs(y_true - y_pred) / (np.abs(y_true) + eps)


In [21]:

import pickle

with open('errors_for_wilcoxon/sr-ri.pkl', 'rb') as file:
    # Load the pickled data
    (list_vr, list_vr_est_srri, list_vbl, list_vbl_est_srri) = pickle.load(file)

with open('errors_for_wilcoxon/sr_vanilla.pkl', 'rb') as file:
    # Load the pickled data
    (list_vr, list_vr_est_sr, list_vbl, list_vbl_est_sr) = pickle.load(file)

with open('errors_for_wilcoxon/xgb.pkl', 'rb') as file:
    # Load the pickled data
    (list_vr, list_vr_est_xgb, list_vbl, list_vbl_est_xgb) = pickle.load(file)

with open('errors_for_wilcoxon/rf.pkl', 'rb') as file:
    # Load the pickled data
    (list_rf, list_vr_est_rf, list_rf, list_vbl_est_rf) = pickle.load(file)

In [161]:

def remove_extream_values(errors_A, errors_B):
    #max_valA = np.max(errors_A)
    max_posA = np.argmax(errors_A)

    #max_valB = np.max(errors_B)
    max_posB = np.argmax(errors_B)

    #print(max_posA)
    #print(max_posB)
    if max_posA == max_posB:
        #print("=")
        errors_A = np.delete(errors_A, max_posA)
        errors_B = np.delete(errors_B, max_posA)
    elif max_posA < max_posB:
        #print("<")
        errors_A = np.delete(errors_A, max_posA)
        errors_B = np.delete(errors_B, max_posA)
        #print(errors_A)
        #print(errors_B)

        errors_A = np.delete(errors_A, max_posB - 1)
        errors_B = np.delete(errors_B, max_posB - 1)
    else:
        #print(">")
        # max_posA > max_posA:
        errors_A = np.delete(errors_A, max_posB)
        errors_B = np.delete(errors_B, max_posB)
        errors_A = np.delete(errors_A, max_posA - 1)
        errors_B = np.delete(errors_B, max_posA - 1)
    return (errors_A, errors_B)




In [175]:
def calculate_errors(list_vr, list_vr_est, list_vbl, list_vbl_est, rev=False):
    rmse_list = []
    mae_list = []
    r2_list = []
    id_list = 0

    eps = 0.000001

    for id_list in range(len(list_vr)):
        id_arr = 0
        while list_vr[id_list][id_arr] < eps and list_vr_est[id_list][id_arr] < eps and id_arr < list_vr[id_list].shape[0]:
                id_arr += 1

        rmse_list.append(rmse(list_vr[id_list][id_arr:list_vr[id_list].shape[0]], list_vr_est[id_list][id_arr:list_vr[id_list].shape[0]]))
        mae_list.append(mae(list_vr[id_list][id_arr:list_vr[id_list].shape[0]], list_vr_est[id_list][id_arr:list_vr[id_list].shape[0]]))
        r2_list.append(r2_score(list_vr[id_list][id_arr:list_vr[id_list].shape[0]], list_vr_est[id_list][id_arr:list_vr[id_list].shape[0]]))

    rmse_l = rmse_list
    if rev:
        rmse_list_rev = remove_extream_values(rmse_list, rmse_list)
        print("RMSE = " + str(np.round(np.mean(rmse_list_rev),3)) + " ± " + str(np.round(np.std(rmse_list_rev),3)))
    else:
        print("RMSE = " + str(np.round(np.mean(rmse_list),3)) + " ± " + str(np.round(np.std(rmse_list),3)))
    mae_l = mae_list
    if rev:
        mae_list_rev = remove_extream_values(mae_list, mae_list)
        print("MAE = " + str(np.round(np.mean(mae_list_rev),3)) + " ± " + str(np.round(np.std(mae_list_rev),3)))
    else:
        print("MAE = " + str(np.round(np.mean(mae_list),3)) + " ± " + str(np.round(np.std(mae_list),3)))
    r2_l = r2_list
    if rev:
        r2_list_rev = remove_extream_values(r2_list, r2_list)
        print("R2 = " + str(np.round(np.mean(r2_list_rev),3)) + " ± " + str(np.round(np.std(r2_list_rev),3)))
    else:
        print("R2 = " + str(np.round(np.mean(r2_list),3)) + " ± " + str(np.round(np.std(r2_list),3)))

    re_vbl = relative_error(list_vbl, list_vbl_est)
    #rmse_vbl = rmse(list_vbl, list_vbl_est)

    print("RE vbl = " + str(np.round(np.mean(re_vbl),3)) + " ± " + str(np.round(np.std(re_vbl),3)))
    rmse_vbl = rmse(list_vbl, list_vbl_est)
    print("RMSE vbl = " + str(np.round(rmse(list_vbl, list_vbl_est),3)))
    mae_vbl = mae(list_vbl, list_vbl_est)
    print("MAE vbl = " + str(np.round(mae(list_vbl, list_vbl_est),3)))
    r2_vbl = relative_error(list_vbl, list_vbl_est)
    print("R2 vbl = " + str(np.round(r2_score(list_vbl, list_vbl_est),3)))
    return (rmse_l, mae_l, r2_l, re_vbl, se(list_vbl, list_vbl_est), ae(list_vbl, list_vbl_est))

(rmse_l_srri, mae_l_srri, r2_l_srri, re_vbl_srri, se_vbl_srri, me_vbl_srri) = calculate_errors(list_vr, list_vr_est_srri, list_vbl, list_vbl_est_srri)
print("################################\n")
(rmse_l_sr, mae_l_sr, r2_l_sr, re_vbl_sr, se_vbl_sr, me_vbl_sr) = calculate_errors(list_vr, list_vr_est_sr, list_vbl, list_vbl_est_sr)
print("################################\n")
(rmse_l_xgb, mae_l_xgb, r2_l_xgb, re_vbl_xgb, se_vbl_xgb, me_vbl_xgb) = calculate_errors(list_vr, list_vr_est_xgb, list_vbl, list_vbl_est_xgb)
print("################################\n")
(rmse_l_rf, mae_l_rf, r2_l_rf, re_vbl_rf, se_vbl_rf, me_vbl_rf) = calculate_errors(list_vr, list_vr_est_rf, list_vbl, list_vbl_est_rf)
print("################################\n")
print("################################\n")

(_foo, _foo, _foo,_foo, _foo,_foo) = calculate_errors(list_vr, list_vr_est_srri, list_vbl, list_vbl_est_srri, True)
print("################################\n")
(_foo, _foo, _foo,_foo, _foo,_foo) = calculate_errors(list_vr, list_vr_est_sr, list_vbl, list_vbl_est_sr, True)
print("################################\n")
(_foo, _foo, _foo,_foo, _foo,_foo) = calculate_errors(list_vr, list_vr_est_xgb, list_vbl, list_vbl_est_xgb, True)
print("################################\n")
(_foo, _foo, _foo,_foo, _foo,_foo) = calculate_errors(list_vr, list_vr_est_rf, list_vbl, list_vbl_est_rf, True)


RMSE = 52.504 ± 47.586
MAE = 46.248 ± 44.058
R2 = 0.894 ± 0.176
RE vbl = 0.028 ± 0.029
RMSE vbl = 25.678
MAE vbl = 16.641
R2 vbl = 0.934
################################

RMSE = 51.656 ± 38.218
MAE = 47.071 ± 37.509
R2 = 0.908 ± 0.116
RE vbl = 0.043 ± 0.055
RMSE vbl = 32.697
MAE vbl = 20.375
R2 vbl = 0.894
################################

RMSE = 93.599 ± 38.906
MAE = 81.782 ± 41.549
R2 = 0.776 ± 0.202
RE vbl = 0.08 ± 0.034
RMSE vbl = 42.809
MAE vbl = 39.125
R2 vbl = 0.818
################################

RMSE = 93.709 ± 40.622
MAE = 82.087 ± 43.911
R2 = 0.771 ± 0.213
RE vbl = 0.08 ± 0.034
RMSE vbl = 42.809
MAE vbl = 39.125
R2 vbl = 0.818
################################

################################

RMSE = 37.575 ± 28.369
MAE = 32.585 ± 26.924
R2 = 0.88 ± 0.184
RE vbl = 0.028 ± 0.029
RMSE vbl = 25.678
MAE vbl = 16.641
R2 vbl = 0.934
################################

RMSE = 41.095 ± 27.876
MAE = 36.958 ± 28.1
R2 = 0.896 ± 0.12
RE vbl = 0.043 ± 0.055
RMSE vbl = 32.697
MAE vbl = 20

In [163]:

import numpy as np
from scipy.stats import wilcoxon

# Przykładowe błędy aproksymacji z dwóch metod
# (w prawdziwej analizie używasz swoich obserwacji)
errors_A = np.array(rmse_l_srri)
#rmse_l_sr
#rmse_l_srri
#rmse_l_xgb
#rmse_l_rf
errors_B = np.array(rmse_l_xgb)
#errors_A = errors_A[1:len(errors_A)-1]
#errors_B = errors_B[1:len(errors_B)-1]
print(errors_A)
print(errors_B)
(errors_A, errors_B) = remove_extream_values(errors_A, errors_B)

print(errors_A)
print(errors_B)

# Test par Wilcoxona:
# Czy metoda A ma istotnie mniejszy błąd niż B?
stat, p_value = wilcoxon(errors_A, errors_B, alternative='less')

print("Statystyka W:", stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Wniosek: Metoda A jest statystycznie lepsza (mniejszy błąd) od metody B.")
else:
    print("Wniosek: Brak istotnej przewagi metody A nad B.")


[157.00923286 106.77217219  26.42138623  27.4548562   23.05240212
  26.15399849  30.79747031  22.36976511]
[ 55.79822695 180.34830124  92.04457768  94.0257316   86.27656808
 120.93407851  54.61869188  64.74865512]
[26.42138623 27.4548562  23.05240212 26.15399849 30.79747031 22.36976511]
[ 92.04457768  94.0257316   86.27656808 120.93407851  54.61869188
  64.74865512]
Statystyka W: 0.0
p-value: 0.015625
Wniosek: Metoda A jest statystycznie lepsza (mniejszy błąd) od metody B.


In [178]:
import math
def calculate_wilcoxon_test(err_list, headers_names):
    tab_string = "Model 1 / Model 2 & "
    #err_list = [np.array(rmse_l_srri), np.array(rmse_l_sr), np.array(rmse_l_rf), np.array(rmse_l_xgb)]
    for h_id in range(len(headers_names)):
        if h_id > 0:
            tab_string += " & "
        tab_string += str(headers_names[h_id])
    tab_string += "\\\\\n\\hline\n"
    for errA_id in range(len(err_list)):
        tab_string += str(headers_names[errA_id]) + " & "
        for errB_id in range(len(err_list)):
            errA = np.copy(err_list[errA_id])
            errB = np.copy(err_list[errB_id])
            (errA, errB) = remove_extream_values(errA, errB)
            #print(errA)
            #print(errB)
            stat, p_value = wilcoxon(errA, errB, alternative='less')
            if errB_id > 0:
                tab_string += " & "
            tab_string += str(round(p_value,3))
        tab_string += "\\\\\n"
    return tab_string



In [179]:
with open("tables/wilcoxon_rmse_curve.txt", "w", encoding="utf-8") as f:
    f.write(calculate_wilcoxon_test([np.array(rmse_l_sr), np.array(rmse_l_srri), np.array(rmse_l_rf), np.array(rmse_l_xgb)], ["SR", "SR R-I", "RF", "XGBoost"]))
with open("tables/wilcoxon_mae_curve.txt", "w", encoding="utf-8") as f:
    f.write(calculate_wilcoxon_test([np.array(rmse_l_sr), np.array(rmse_l_srri), np.array(mae_l_rf), np.array(mae_l_xgb)], ["SR", "SR R-I", "RF", "XGBoost"]))
with open("tables/wilcoxon_r2_curve.txt", "w", encoding="utf-8") as f:
    f.write(calculate_wilcoxon_test([np.array(rmse_l_sr), np.array(rmse_l_srri), np.array(r2_l_rf), np.array(r2_l_xgb)], ["SR", "SR R-I", "RF", "XGBoost"]))
with open("tables/wilcoxon_re_vbl.txt", "w", encoding="utf-8") as f:
    f.write(calculate_wilcoxon_test([np.array(re_vbl_sr), np.array(re_vbl_srri), np.array(re_vbl_rf), np.array(re_vbl_xgb)], ["SR", "SR R-I", "RF", "XGBoost"]))
with open("tables/wilcoxon_se_vbl.txt", "w", encoding="utf-8") as f:
    f.write(calculate_wilcoxon_test([np.array(se_vbl_sr), np.array(se_vbl_srri), np.array(se_vbl_rf), np.array(se_vbl_xgb)], ["SR", "SR R-I", "RF", "XGBoost"]))
with open("tables/wilcoxon_me_vbl.txt", "w", encoding="utf-8") as f:
    f.write(calculate_wilcoxon_test([np.array(me_vbl_sr), np.array(me_vbl_srri), np.array(me_vbl_rf), np.array(me_vbl_xgb)], ["SR", "SR R-I", "RF", "XGBoost"]))

In [182]:
#[18.0, 16.0, 14.0, 12.0, 10.0, 8.0, 6.0, 4.0]
rmse_l_srri

[np.float64(157.00923285914504),
 np.float64(106.772172190108),
 np.float64(26.421386231163147),
 np.float64(27.454856198423382),
 np.float64(23.052402122126228),
 np.float64(26.153998488418893),
 np.float64(30.797470313930575),
 np.float64(22.3697651067227)]